In [16]:
BANNER = \
"""
# ----------------------------------------------------------------------
# PTS | Paulo Trigo Silva
# mLDm | MoP
# v01
# ----------------------------------------------------------------------
"""

In [17]:
# ------------------------------------------------------------
# container runtime
# ------------------------------------------------------------

# note:
# before continuing, make sure that 00_penguins_db_boot.py
# is configured for the container runtime that you are using
#
# recall that 00_penguins_db_boot.py is your working copy of
# _mLDm_00_penguins_db_boot.py
#
# in that file, exactly one of these alternatives must be active:
#
# RUNTIME = "podman"
# # RUNTIME = "docker"
#
# or:
#
# RUNTIME = "podman"
RUNTIME = "docker"

In [18]:
# ------------------------------------------------------------
# bootstrap the initial PostgreSQL database
# ------------------------------------------------------------

from pathlib import Path
import subprocess
import sys

# locate the bootstrap script in the current code folder
boot_script = Path("00_penguins_db_boot.py").resolve()

# recreate and populate the initial database
subprocess.run( [sys.executable, str(boot_script)], check=True )

print("PTS | initial database successfully created")

PTS | initial database successfully created


In [19]:
# ------------------------------------------------------------
# connect DuckDB to the initial PostgreSQL database
# ------------------------------------------------------------

import duckdb

# create an in-memory DuckDB connection
db = duckdb.connect()

# install and load the DuckDB PostgreSQL extension
# the INSTALL statement is required only the first time the extension is used
db.sql("""
INSTALL postgres;
LOAD postgres;
""")

# attach the PostgreSQL database to DuckDB
# the database will be accessible through the name "pg"
db.sql("""
ATTACH
'host=localhost port=5432 dbname=mldm user=mldm password=mldm'
AS pg
(TYPE postgres);
""")

print("PTS | DuckDB connected to PostgreSQL")

PTS | DuckDB connected to PostgreSQL


In [20]:
# ------------------------------------------------------------
# import the collected csv into a staging table in postgresql
# ------------------------------------------------------------

from pathlib import Path

# locate the input file independently of the operating system
csv_file = Path("../01_input/_mLDm_00_penguins_data_collected.csv").resolve()

# create or replace the staging table in postgresql
# use a POSIX-style path for portability inside the SQL query
db.sql(f"""
CREATE OR REPLACE TABLE pg.public.collected_stage AS
SELECT *
FROM read_csv(
    '{csv_file.as_posix()}',
    columns={{
        'id_penguin': 'INTEGER',
        'date_time': 'TIMESTAMP',
        'study_name': 'VARCHAR',
        'sample_number': 'INTEGER',
        'species': 'VARCHAR',
        'island': 'VARCHAR',
        'bill_length_mm': 'DOUBLE',
        'bill_depth_mm': 'DOUBLE',
        'flipper_length_mm': 'DOUBLE',
        'body_mass_g': 'DOUBLE',
        'sex': 'VARCHAR',
        'latitude': 'DOUBLE',
        'longitude': 'DOUBLE',
        'name_batch': 'VARCHAR'
    }},
    header=True
);
""")

# verify the number of imported rows
db.sql("""
SELECT COUNT(*) AS n_rows
FROM pg.public.collected_stage;
""").show()

┌────────┐
│ n_rows │
│ int64  │
├────────┤
│    324 │
└────────┘



In [21]:
# ------------------------------------------------------------
# inspect the staging table
#
# note:
# you may also use pgAdmin to inspect:
# - the database structure, and
# - data interactively
# ------------------------------------------------------------

# inspect the staging schema
db.sql("""
DESCRIBE pg.public.collected_stage;
""").show()

# inspect some attributes of a small sample of collected observations
db.sql("""
SELECT id_penguin,
       date_time,
       study_name,
       sample_number,
       species,
       island,
       bill_length_mm,
       latitude,
       name_batch
FROM pg.public.collected_stage
LIMIT 10;
""").show()

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id_penguin        │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ date_time         │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ study_name        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ sample_number     │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ species           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ island            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ bill_length_mm    │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ bill_depth_mm     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ flipper_length_mm │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ body_mass_

In [22]:
# ------------------------------------------------------------
# compare collected data with the existing database
# ------------------------------------------------------------

# count collected observations and distinct penguins
db.sql("""
SELECT
     COUNT(*) AS n_observations,
     COUNT(DISTINCT(id_penguin)) AS n_penguins
FROM pg.public.collected_stage;
""").show()
# expected result:
# n_observations = 324
# n_penguins = 322


# count the number of penguins already in the database
db.sql("""
SELECT  COUNT( DISTINCT c.id_penguin ) AS n_db_existing_penguins
FROM pg.public.collected_stage AS c JOIN pg.public.penguin AS p
     ON c.id_penguin = p.id;
""").show()
# expected result:
# n_db_existing_penguins = 8

┌────────────────┬────────────┐
│ n_observations │ n_penguins │
│     int64      │   int64    │
├────────────────┼────────────┤
│            324 │        322 │
└────────────────┴────────────┘

┌────────────────────────┐
│ n_db_existing_penguins │
│         int64          │
├────────────────────────┤
│                      8 │
└────────────────────────┘



In [23]:
# ------------------------------------------------------------
# identify penguins already present in the database and their collected species
# ------------------------------------------------------------

db.sql("""
SELECT
     DISTINCT(c.id_penguin), species
FROM pg.public.collected_stage AS c JOIN pg.public.penguin AS p
     ON c.id_penguin = p.id
ORDER BY c.id_penguin;
""").show()

# expected result:
# id_penguin   species
# 9            Adelie
# 15           Adelie
# 25           Adelie
# 145          Adelie
# 155          Adelie
# 173          Adelie
# 186          Adelie
# 292          Adelie

┌────────────┬─────────┐
│ id_penguin │ species │
│   int32    │ varchar │
├────────────┼─────────┤
│          9 │ Adelie  │
│         15 │ Adelie  │
│         25 │ Adelie  │
│        145 │ Adelie  │
│        155 │ Adelie  │
│        173 │ Adelie  │
│        186 │ Adelie  │
│        292 │ Adelie  │
└────────────┴─────────┘



In [24]:
# ------------------------------------------------------------
# check species consistency for penguins already in the database
# ------------------------------------------------------------

# compare the species in the collected data
# with the species already stored in PENGUIN
db.sql("""
SELECT COUNT(*) AS n_inconsistent
FROM pg.public.collected_stage AS c JOIN pg.public.penguin AS p
     ON c.id_penguin = p.id
WHERE c.species not in (select name from pg.public.species);
""").show()

# expected result:
# n_inconsistent = 0

┌────────────────┐
│ n_inconsistent │
│     int64      │
├────────────────┤
│              0 │
└────────────────┘



In [25]:
# ------------------------------------------------------------
# create a view with penguins not yet present in the database
# ------------------------------------------------------------

# hint:
# consider using a LEFT JOIN (or RIGHT JOIN)
# to preserve all rows from one side and identify those without a match
# a LEFT (or RIGHT) JOIN preserves all rows from the table on its left (or right)
# while adding matching information from the table on its right (or left)


db.sql("""
CREATE OR REPLACE TEMP VIEW new_penguins AS
SELECT DISTINCT c.id_penguin
FROM pg.public.collected_stage AS c
     LEFT JOIN pg.public.penguin AS p
     ON c.id_penguin = p.id
WHERE c.id_penguin not in (select id from pg.public.penguin);
""")

db.sql("""
SELECT COUNT(*) AS n_new_penguins
FROM new_penguins;
""").show()

# expected result:
# n_new_penguins = 314

┌────────────────┐
│ n_new_penguins │
│     int64      │
├────────────────┤
│            314 │
└────────────────┘



In [26]:
# ------------------------------------------------------------
# insert new penguins into the database
# ------------------------------------------------------------

# count the penguins that are about to be inserted
n_inserted = db.sql("""
SELECT COUNT(*)
FROM new_penguins;
""").fetchone()[0]

# insert the identified new penguins
db.sql("""
INSERT INTO pg.public.penguin ( id, name_species, sex )
SELECT DISTINCT c.id_penguin, c.species, c.sex
FROM pg.public.collected_stage AS c JOIN new_penguins AS n
     ON n.id_penguin = c.id_penguin;
""")

print("inserted penguins:", n_inserted)

# expected result:
# inserted penguins: 314

inserted penguins: 314


In [27]:
# ------------------------------------------------------------
# create a view with observations not yet present in the database
# ------------------------------------------------------------

# note:
# table PENGUIN stores the identity and stable attributes of each penguin
# table OBSERVATION stores individual observations made at particular date_time values
#
# therefore, a penguin may already exist in PENGUIN while a newly collected
# observation of that same penguin is not yet stored in OBSERVATION

# hint:
# consider using the same join strategy applied above


db.sql("""
CREATE OR REPLACE TEMP VIEW new_observations AS
SELECT c.id_penguin, c.date_time
FROM pg.public.collected_stage AS c
     left JOIN pg.public.observation AS o
     ON c.id_penguin = o.id_penguin and c.date_time = o.date_time
WHERE o.date_time is NULL or o.id_penguin is NULL;
""")

# count the observations that still need to be inserted
db.sql("""
SELECT COUNT(*) AS n_new_observations
FROM new_observations;
""").show()

# expected result:
# n_new_observations = 324

┌────────────────────┐
│ n_new_observations │
│       int64        │
├────────────────────┤
│                324 │
└────────────────────┘



In [28]:
# ------------------------------------------------------------
# insert new observations into the database
# ------------------------------------------------------------

# count the observations that are about to be inserted
n_inserted = db.sql("""
SELECT COUNT(*)
FROM new_observations;
""").fetchone()[0]

# insert the identified new observations
db.sql("""
INSERT INTO pg.public.observation (
    id_penguin,
    date_time,
    name_study,
    name_island,
    name_batch,
    sample_number,
    latitude,
    longitude,
    bill_length_mm,
    bill_depth_mm,
    flipper_length_mm,
    body_mass_g
)
SELECT
    c.id_penguin,
    c.date_time,
    c.study_name,
    c.island,
    c.name_batch,
    c.sample_number,
    c.latitude,
    c.longitude,
    c.bill_length_mm,
    c.bill_depth_mm,
    c.flipper_length_mm,
    c.body_mass_g
FROM pg.public.collected_stage AS c JOIN new_observations AS n
     ON  n.id_penguin = c.id_penguin
     AND n.date_time = c.date_time;
""")

print("inserted observations:", n_inserted)

# expected result:
# inserted observations: 324

inserted observations: 324


In [29]:
# ------------------------------------------------------------
# validate the integrated database
# ------------------------------------------------------------

# check the number of penguins and observations after integration
db.sql("""
SELECT (SELECT COUNT(*) FROM pg.public.penguin) AS n_penguins,
       (SELECT COUNT(*) FROM pg.public.observation) AS n_observations;
""").show()


# expected result:
# n_penguins = 334
# n_observations = 344

┌────────────┬────────────────┐
│ n_penguins │ n_observations │
│   int64    │     int64      │
├────────────┼────────────────┤
│        334 │            344 │
└────────────┴────────────────┘



In [30]:
# ------------------------------------------------------------
# remove staging objects no longer required
# ------------------------------------------------------------

# remove the temporary views used during integration
db.sql("""
DROP VIEW IF EXISTS new_observations;
DROP VIEW IF EXISTS new_penguins;
""")

# remove the staging table from postgresql
db.sql("""
DROP TABLE pg.public.collected_stage;
""")

print("PTS | staging objects removed")

PTS | staging objects removed
